# GP Circuit — Post-Hoc Analysis

This notebook loads the **finalized binary-mask circuit** discovered by `gp_llama_kl_budget.py`
and performs a set of interpretability analyses adapted from circuit-interpretability methodology.

## Task: Gender Pronouns (GP)
Given a sentence like *"Diamond stays calm under pressure, doesn't **she**"*,
the model must predict the correct gender pronoun (`she` or `he`) based on the gender-indicative
first name.  Clean and corrupted (name-swapped) sentence pairs allow KL-divergence-based
faithfulness evaluation.

**Terminology:**
- `target` — the correct pronoun (`she` or `he`) matching the gendered name
- `distractor` — the opposite pronoun
- `pred_pos` — token position just before the pronoun (where model outputs logits)
- `name_pos` — position of the gendered name in the sentence

## Analysis sections
1. Circuit Inventory — surviving heads / neurons / blocks
2. Attention Patterns — where do heads attend when predicting the pronoun?
3. Direct Logit Attribution (DLA) — per-component contribution to target vs distractor logit
4. Head Role Classification — Name-Mover, Pronoun-Promoter, etc.
5. OV Circuit — what does each head write to the residual stream?
6. Head Ablation Study — how critical is each surviving head?
7. Logit Lens — how does the prediction evolve through layers?
8. Individual Neuron Analysis — vocab projection, input sensitivity, activation correlation
9. End-to-End Circuit Story — synthesised narrative

In [ ]:
import os, sys, pickle
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

# ---- make project root importable ----
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'models')):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from posthoc_analysis.analysis_utils import (
    load_circuit,
    get_surviving_heads,
    get_surviving_mlp_blocks,
    get_surviving_attn_blocks,
    get_surviving_mlp_neurons,
    circuit_summary_table,
    get_attention_patterns,
    compute_dla,
    dla_io_vs_s,
    compute_ov_top_tokens,
    ablate_head_and_eval,
    ablate_mlp_block_and_eval,
    compute_logit_lens,
    find_name_positions_batch,
)
from dataset.gp_llama import GPDatasetLlama, run_evaluation

print('Imports OK')

## Configuration — set `SAVE_DIR` to your GP checkpoint folder

In [ ]:
SAVE_DIR   = '/home/exouser/circuit_pruning/results/gp_kl_budget_final_1B_with_blocks_0.7kl'  # <-- change to your GP run
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16      # reduce if OOM
N_ANALYSIS = 100     # samples for attention-pattern / DLA analysis (subset for speed)

print(f'Device: {DEVICE}  |  Save dir: {SAVE_DIR}')

In [ ]:
# Load models (uses cached weights)
circuit_model, full_model, tokenizer, run_config = load_circuit(SAVE_DIR, device=DEVICE)

print(f"\nModel      : {run_config['model']}")
print(f"KL budget  : {run_config['kl_budget']}")
print(f"Circuit is in final binary-mask mode ✓")

In [ ]:
# Load test data
with open(os.path.join(SAVE_DIR, 'test_data.pkl'), 'rb') as f:
    test_data = pickle.load(f)

test_dataset    = GPDatasetLlama(test_data, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Also load binary masks for quick gate inspection
binary_masks = torch.load(os.path.join(SAVE_DIR, 'binary_masks.pt'), map_location='cpu')

print(f'Test samples: {len(test_data)}  |  Valid after tokenisation filter: {len(test_dataset)}')

---
## Section 1 — Circuit Inventory

How many heads / neurons survived the KL-budget pruning?  
This is the basic *what* before we ask *why*.

In [ ]:
summary = circuit_summary_table(circuit_model)

n_total_heads   = circuit_model.config.num_attention_heads * len(circuit_model.model.layers)
n_surv_heads    = summary['n_heads'].sum()
n_surv_attn_blk = summary['attn_block'].sum()
n_surv_mlp_blk  = summary['mlp_block'].sum()
n_surv_hid      = summary['n_hid_neurons'].sum()
n_total_hid     = circuit_model.config.intermediate_size * len(circuit_model.model.layers)

print(f"Surviving attention heads  : {n_surv_heads} / {n_total_heads}  ({n_surv_heads/n_total_heads*100:.1f}%)")
print(f"Surviving attn blocks      : {n_surv_attn_blk} / {len(circuit_model.model.layers)}")
print(f"Surviving MLP blocks       : {n_surv_mlp_blk} / {len(circuit_model.model.layers)}")
print(f"Surviving MLP hidden neurons: {n_surv_hid} / {n_total_hid}  ({n_surv_hid/n_total_hid*100:.1f}%)")
print()
print(summary[['layer','attn_block','n_heads','head_pct','mlp_block','n_hid_neurons','hid_pct']].to_string(index=False))

In [ ]:
# Visual: surviving heads per layer (heatmap)
num_heads = circuit_model.config.num_attention_heads
n_layers  = len(circuit_model.model.layers)
surv_heads_dict = get_surviving_heads(circuit_model)

head_grid = np.zeros((n_layers, num_heads))
for l, heads in surv_heads_dict.items():
    for h in heads:
        head_grid[l, h] = 1

fig, ax = plt.subplots(figsize=(max(8, num_heads * 0.5 + 2), max(4, n_layers * 0.4 + 2)))
sns.heatmap(head_grid, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=[f'H{h}' for h in range(num_heads)],
            yticklabels=[f'L{l}' for l in range(n_layers)],
            linewidths=0.5, cbar=False)
ax.set_title(f'Surviving attention heads ({n_surv_heads}/{n_total_heads} = {n_surv_heads/n_total_heads*100:.1f}%)')
ax.set_xlabel('Head index')
ax.set_ylabel('Layer')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_head_mask.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visual: surviving MLP hidden neurons per layer
surv_neurons = get_surviving_mlp_neurons(circuit_model)
layers_axis  = list(range(n_layers))
hid_pcts     = [len(surv_neurons[l][0]) / circuit_model.config.intermediate_size * 100 for l in layers_axis]
surv_mlp_blks = get_surviving_mlp_blocks(circuit_model)

colors = ['steelblue' if surv_mlp_blks.get(l, True) else 'lightgrey' for l in layers_axis]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(layers_axis, hid_pcts, color=colors)
ax.set_xlabel('Layer')
ax.set_ylabel('% MLP hidden neurons surviving')
ax.set_title('MLP hidden neuron survival per layer  (grey = whole MLP block pruned)')
ax.set_xticks(layers_axis)
ax.set_xticklabels([f'L{l}' for l in layers_axis], fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_mlp_neurons.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — Attention Patterns on GP Examples

We compute **approximate** attention patterns by hooking the Q and K projections
and computing Q·Kᵀ / √d without RoPE (qualitatively accurate for identifying which
positions a head attends to).

**GP-specific question:** Does each surviving head attend to the **gendered name position**
when predicting the pronoun?  
- High attention to name → head may be identifying the gender context
- Combined with DLA, this distinguishes name-movers from other role types

In [ ]:
# Grab the first N_ANALYSIS test examples
analysis_batch = next(iter(DataLoader(test_dataset, batch_size=N_ANALYSIS, shuffle=False)))

# ---- Prediction position ----
# prefix_length is the number of tokens in the sentence prefix (including BOS).
# The model predicts the pronoun at position prefix_length, so we read logits
# at pred_pos = prefix_length - 1.
pred_pos = analysis_batch['prefix_length'] - 1    # (B,)  0-indexed

# ---- Target and distractor token IDs ----
target_ids     = analysis_batch['target_token']      # (B,)  scalar per sample
distractor_ids = analysis_batch['distractor_token']  # (B,)

# ---- Find gendered name positions in tokenised input ----
# Access raw processed data to get the name string for each sample
name_token_ids_list = []
for b in range(len(pred_pos)):
    raw = test_dataset.processed_data[b]
    name = raw.get('name', '')
    # Encode with a space prefix (as the name appears mid-sentence)
    toks = tokenizer.encode(' ' + name, add_special_tokens=False)
    name_token_ids_list.append(toks[0] if toks else -1)

name_token_ids = torch.tensor(name_token_ids_list, dtype=torch.long)   # (B,)

# Search for name token positions in input (before pred_pos)
# search_end = pred_pos + 1 means we include positions [bos_offset, pred_pos]
name_positions_list = find_name_positions_batch(
    analysis_batch['input_ids'],
    name_token_ids,
    pred_pos + 1,   # exclusive upper bound
)  # list[list[int]]

# Sanity check
name_counts = [len(p) for p in name_positions_list]
print(f'Name occurrences found: min={min(name_counts)}  max={max(name_counts)}  mean={sum(name_counts)/len(name_counts):.2f}')
print(f'Samples with name found: {sum(1 for c in name_counts if c > 0)} / {len(name_counts)}')
print(f'Ex 0: pred_pos={pred_pos[0].item()}  name_pos={name_positions_list[0]}')
print(f'      Sentence: {tokenizer.decode(analysis_batch["input_ids"][0].tolist(), skip_special_tokens=False)[:120]}')
print(f'      Target pronoun: "{tokenizer.decode([target_ids[0].item()])}"')

# For downstream analyses: use last name occurrence per sample
B = len(name_positions_list)
name_pos = torch.tensor([ps[-1] if ps else pred_pos[b].item()-1
                          for b, ps in enumerate(name_positions_list)])   # (B,)

print(f'\nComputing attention patterns for {N_ANALYSIS} examples ...')
attn_patterns = get_attention_patterns(
    circuit_model,
    analysis_batch['input_ids'],
    analysis_batch['attention_mask'],
    analysis_batch['corrupted_input_ids'],
    device=DEVICE,
)
print(f'Done. Patterns computed for {len(attn_patterns)} layers.')

In [ ]:
# Compute GP-specific attention scores at prediction position
# For each surviving head: attention weight from pred_pos → name position
name_attn_scores = {}   # {layer: (B, H)}  attention to name position
seq_len = analysis_batch['input_ids'].shape[1]

for l, patt in attn_patterns.items():
    B_l, H, S, _ = patt.shape
    attn_to_name = torch.zeros(B_l, H)

    for b in range(B_l):
        pred_p   = pred_pos[b].item()
        name_ps  = name_positions_list[b]
        if name_ps and pred_p < S:
            for np_ in name_ps:
                if np_ < S:
                    attn_to_name[b] += patt[b, :, pred_p, np_]
            attn_to_name[b] /= len(name_ps)

    name_attn_scores[l] = attn_to_name  # (B, H)

# Mean over batch
mean_name_attn = {l: v.mean(0).numpy() for l, v in name_attn_scores.items()}  # {layer: (H,)}

In [ ]:
# Plot: per surviving head — attention to name position  (2-D heatmap: layer × head)
surviving_heads_dict = get_surviving_heads(circuit_model)
surv_head_list = [(l, h) for l, heads in sorted(surviving_heads_dict.items()) for h in heads]

if not surv_head_list:
    print('No surviving heads to plot.')
else:
    all_layers = sorted({l for l, _ in surv_head_list})
    all_heads  = sorted({h for _, h in surv_head_list})
    nl = max(all_layers) + 1
    nh = max(all_heads) + 1
    surv_set = set(surv_head_list)

    name_grid = np.full((nl, nh), np.nan)
    for l in all_layers:
        if l not in mean_name_attn:
            continue
        for h in range(mean_name_attn[l].shape[0]):
            if (l, h) in surv_set:
                name_grid[l, h] = mean_name_attn[l][h]

    vmax_n = np.nanmax(name_grid) if not np.all(np.isnan(name_grid)) else 0.3

    fig, ax = plt.subplots(figsize=(max(8, nh * 0.55 + 2), max(4, nl * 0.55 + 2)))
    masked = np.ma.array(name_grid, mask=np.isnan(name_grid))
    im = ax.imshow(masked, cmap='Blues', vmin=0, vmax=vmax_n,
                   aspect='auto', interpolation='nearest')
    ax.set_xticks(range(nh)); ax.set_xticklabels([f'H{h}' for h in range(nh)], fontsize=7)
    ax.set_yticks(range(nl)); ax.set_yticklabels([f'L{l}' for l in range(nl)], fontsize=7)
    ax.set_xlabel('Head'); ax.set_ylabel('Layer')
    ax.set_title('Surviving heads: attention to gendered name position at pred_pos\n(grey = pruned head)')
    for l in all_layers:
        for h in range(nh):
            v = name_grid[l, h]
            if not np.isnan(v):
                ax.text(h, l, f'{v:.2f}', ha='center', va='center',
                        fontsize=6, color='white' if v > vmax_n * 0.6 else 'black')
    grey = np.zeros((*name_grid.shape, 4))
    grey[np.isnan(name_grid)] = [0.85, 0.85, 0.85, 1.0]
    ax.imshow(grey, aspect='auto', interpolation='nearest',
              extent=[-0.5, nh - 0.5, nl - 0.5, -0.5])
    plt.colorbar(im, ax=ax, shrink=0.7, label='mean attention weight to name pos')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'posthoc_attn_name.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# For a specific head of interest, plot the full attention pattern
# Change to any (layer, head) from surv_head_list:
VIZ_LAYER, VIZ_HEAD = surv_head_list[0] if surv_head_list else (0, 0)

if VIZ_LAYER in attn_patterns:
    # Average over batch examples that have a valid name position
    valid = [b for b in range(N_ANALYSIS) if name_positions_list[b]]
    if valid:
        patt_mean = attn_patterns[VIZ_LAYER][valid, VIZ_HEAD].mean(0).numpy()  # (S,)
        pred_p_eg = pred_pos[valid[0]].item()

        fig, ax = plt.subplots(figsize=(14, 3))
        ax.bar(range(len(patt_mean)), patt_mean, color='steelblue', alpha=0.7)
        if name_positions_list[valid[0]]:
            for np_ in name_positions_list[valid[0]]:
                ax.axvline(np_, color='green', linewidth=2, alpha=0.8, label='name pos' if np_ == name_positions_list[valid[0]][0] else '')
        ax.axvline(pred_p_eg, color='red', linewidth=2, linestyle='--', label='pred pos')
        ax.set_xlim(0, pred_p_eg + 5)
        ax.set_xlabel('Token position')
        ax.set_ylabel('Attention weight')
        ax.set_title(f'L{VIZ_LAYER}H{VIZ_HEAD} — attention pattern at pred_pos (mean over {len(valid)} examples)\n'
                     f'Sentence: {tokenizer.decode(analysis_batch["input_ids"][valid[0]].tolist(), skip_special_tokens=True)[:80]}...')
        ax.legend()
        plt.tight_layout()
        plt.show()

---
## Section 3 — Direct Logit Attribution (DLA)

DLA decomposes the final logit into contributions from each component:
$$\text{logit}(t) = W_U \cdot \text{LN}\left(\sum_c v_c\right) \approx \sum_c W_U \cdot \hat{v}_c$$

Using the frozen-scale linearisation of RMSNorm, we attribute a (target − distractor) logit
difference to each attention head and MLP layer.

**GP interpretation:**
- **Positive DLA(target − distractor)** → component promotes the correct pronoun
- **Negative DLA** → component harms the task (may be pruned or serves another role)

In [ ]:
print('Computing DLA (this runs one forward pass with hooks) ...')
dla_result = compute_dla(
    circuit_model,
    analysis_batch['input_ids'],
    analysis_batch['attention_mask'],
    analysis_batch['corrupted_input_ids'],
    pred_positions=pred_pos,
    device=DEVICE,
)
print('DLA done.')

In [ ]:
# Per-head: mean (target DLA − distractor DLA) — positive = helps task
head_dla_diff = {}   # {(layer, head): float}

for l, attn_dla in dla_result['attn'].items():
    # attn_dla: (B, H, vocab)
    io_s, s_s = dla_io_vs_s(attn_dla, target_ids, distractor_ids)
    diff = (io_s - s_s).mean(0).numpy()    # (H,)
    for h in range(diff.shape[0]):
        head_dla_diff[(l, h)] = diff[h]

# Per-MLP-layer: mean (target DLA − distractor DLA)
mlp_dla_diff = {}
for l, mlp_dla in dla_result['mlp'].items():
    io_s, s_s = dla_io_vs_s(mlp_dla, target_ids, distractor_ids)
    mlp_dla_diff[l] = (io_s - s_s).mean().item()

In [ ]:
# Plot DLA for surviving heads (sorted by DLA value)
if surv_head_list:
    dla_vals = [head_dla_diff.get((l, h), 0.0) for (l, h) in surv_head_list]
    colors   = ['steelblue' if v >= 0 else 'tomato' for v in dla_vals]
    labels   = [f'L{l}H{h}' for (l, h) in surv_head_list]

    # Sort by DLA descending
    order = np.argsort(-np.array(dla_vals))

    fig, ax = plt.subplots(figsize=(max(10, len(surv_head_list) * 0.6 + 2), 4))
    ax.bar(range(len(surv_head_list)),
           [dla_vals[i] for i in order],
           color=[colors[i] for i in order])
    ax.set_xticks(range(len(surv_head_list)))
    ax.set_xticklabels([labels[i] for i in order], rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Attention head')
    ax.set_ylabel('DLA (target − distractor logit)')
    ax.set_title('Per-head DLA: contribution to correct pronoun prediction\n(blue = promotes target, red = opposes target)')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'posthoc_head_dla.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot MLP DLA per layer
if mlp_dla_diff:
    ls_mlp = sorted(mlp_dla_diff.keys())
    vals   = [mlp_dla_diff[l] for l in ls_mlp]
    surv_mlp = get_surviving_mlp_blocks(circuit_model)

    fig, ax = plt.subplots(figsize=(10, 3))
    colors_mlp = ['steelblue' if v >= 0 else 'tomato' for v in vals]
    hatches    = ['' if surv_mlp.get(l, True) else '///' for l in ls_mlp]
    for i, (l, v) in enumerate(zip(ls_mlp, vals)):
        ax.bar(i, v, color=colors_mlp[i], hatch=hatches[i], alpha=0.8)
    ax.set_xticks(range(len(ls_mlp)))
    ax.set_xticklabels([f'L{l}' for l in ls_mlp], fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Layer')
    ax.set_ylabel('DLA (target − distractor)')
    ax.set_title('Per-layer MLP DLA  (hatched = whole MLP block pruned)')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'posthoc_mlp_dla.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## Section 4 — Head Role Classification

Combining attention to name position and DLA, we classify each surviving head:

| Role | Attention to name | DLA(target−distractor) |
|------|-------------------|------------------------|
| **Name-Mover** | high (>0.05) | positive (>0.05) |
| **Name-Suppressor** | high (>0.05) | negative (<−0.05) |
| **Pronoun-Promoter** | low | positive (>0.05) |
| **Pronoun-Suppressor** | low | negative (<−0.05) |
| **Other** | — | — |

**Name-Mover** heads are the GP analogue of IOI Name-Mover heads: they attend to the
gendered name and write a representation that promotes the correct pronoun.

In [ ]:
def classify_head_role_gp(name_attn: float, dla_diff: float,
                           name_thresh: float = 0.05, dla_thresh: float = 0.05) -> str:
    if name_attn > name_thresh and dla_diff > dla_thresh:
        return 'Name-Mover'
    elif name_attn > name_thresh and dla_diff < -dla_thresh:
        return 'Name-Suppressor'
    elif dla_diff > dla_thresh:
        return 'Pronoun-Promoter'
    elif dla_diff < -dla_thresh:
        return 'Pronoun-Suppressor'
    else:
        return 'Other'

role_rows = []
for (l, h) in surv_head_list:
    n_attn = float(mean_name_attn[l][h]) if l in mean_name_attn else 0.0

    # DLA for this specific head
    if l in dla_result['attn']:
        attn_dla = dla_result['attn'][l]    # (B, H, vocab)
        tgt_d = attn_dla[:, h, target_ids].diagonal().mean().item()
        dis_d = attn_dla[:, h, distractor_ids].diagonal().mean().item()
    else:
        tgt_d, dis_d = 0.0, 0.0

    role = classify_head_role_gp(n_attn, tgt_d - dis_d)
    role_rows.append({
        'layer': l, 'head': h, 'label': f'L{l}H{h}',
        'name_attn': round(n_attn, 4),
        'target_dla': round(tgt_d, 3),
        'distractor_dla': round(dis_d, 3),
        'dla_diff': round(tgt_d - dis_d, 3),
        'role': role,
    })

role_df = pd.DataFrame(role_rows)
print(role_df[['label','name_attn','target_dla','distractor_dla','dla_diff','role']].to_string(index=False))
print()
print(role_df['role'].value_counts().to_string())

In [ ]:
# Scatter plot: name attention vs DLA(target−distractor) — color-coded by role
ROLE_COLORS = {
    'Name-Mover':       'steelblue',
    'Name-Suppressor':  'navy',
    'Pronoun-Promoter': 'seagreen',
    'Pronoun-Suppressor': 'tomato',
    'Other':            'grey',
}

if not role_df.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    for role, grp in role_df.groupby('role'):
        ax.scatter(grp['name_attn'], grp['dla_diff'],
                   color=ROLE_COLORS.get(role, 'grey'), label=role,
                   s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
        for _, row in grp.iterrows():
            ax.annotate(row['label'], (row['name_attn'], row['dla_diff']),
                        fontsize=7, ha='left', va='bottom')
    ax.axhline(0, color='black', lw=0.8, linestyle='-')
    ax.axvline(0.05, color='black', lw=0.8, linestyle='--', alpha=0.4, label='name thresh')
    ax.axhline(0.05, color='black', lw=0.8, linestyle=':', alpha=0.4, label='DLA thresh')
    ax.axhline(-0.05, color='black', lw=0.8, linestyle=':', alpha=0.4)
    ax.set_xlabel('Mean attention to name position')
    ax.set_ylabel('DLA (target − distractor pronoun logit)')
    ax.set_title('Head role classification: name attention vs DLA')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'posthoc_role_scatter.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## Section 5 — OV Circuit: Unembedding Projection

The OV circuit asks: *given that a head attends to token t, what does it write into the residual stream?*

$$W_{OV}^{(h)} = W_V^{(h)} W_O^{(h)}$$

We project this through the final LayerNorm and unembedding matrix to get vocabulary-space
token promotions for each head.

**GP interpretation:** A Name-Mover head attends to the name.  
If its OV matrix boosts the target pronoun, the `target_copy_score` will be positive.

In [ ]:
ov_rows = []
for (l, h) in tqdm(surv_head_list, desc='OV analysis'):
    top_copy, top_prom, top_supp, copy_score = compute_ov_top_tokens(
        circuit_model, l, h, tokenizer, top_k=10
    )

    # GP-specific copy scores: how much does OV matrix promote target vs distractor pronouns?
    # Average over the analysis batch (different samples may have different target/distractor)
    ov_w = circuit_model.model.layers[l].attn.original_attention
    W_V  = ov_w.v_proj.weight.detach().float().cpu()    # (kv_heads * head_dim, hidden)
    W_O  = ov_w.o_proj.weight.detach().float().cpu()    # (hidden, num_heads * head_dim)
    W_U  = circuit_model.lm_head.weight.detach().float().cpu()  # (vocab, hidden)
    W_E  = circuit_model.model.embed_tokens.weight.detach().float().cpu()  # (vocab, hidden)

    head_dim   = circuit_model.config.hidden_size // circuit_model.config.num_attention_heads
    num_kv     = circuit_model.config.num_key_value_heads
    kv_groups  = circuit_model.config.num_attention_heads // num_kv
    kv_h       = h // kv_groups

    v_slice = W_V[kv_h * head_dim : (kv_h + 1) * head_dim, :]  # (D, hidden)
    o_slice = W_O[:, h * head_dim : (h + 1) * head_dim]         # (hidden, D)
    OV      = o_slice @ v_slice   # (hidden, hidden)

    # Target copy score: when head attends to a name token, how much does it promote target pronoun?
    tgt_scores  = []
    dis_scores  = []
    for b in range(len(target_ids)):
        raw = test_dataset.processed_data[b]
        name = raw.get('name', '')
        name_tok = tokenizer.encode(' ' + name, add_special_tokens=False)
        if not name_tok:
            continue
        embed_name = W_E[name_tok[0]].float()  # (hidden,)
        written    = OV @ embed_name            # (hidden,) — what head writes when attending to name
        tgt_logit  = (W_U[target_ids[b].item()].float() @ written).item()
        dis_logit  = (W_U[distractor_ids[b].item()].float() @ written).item()
        tgt_scores.append(tgt_logit)
        dis_scores.append(dis_logit)

    tgt_copy = float(np.mean(tgt_scores)) if tgt_scores else float('nan')
    dis_copy = float(np.mean(dis_scores)) if dis_scores else float('nan')

    ov_rows.append({
        'layer': l, 'head': h, 'label': f'L{l}H{h}',
        'target_copy_score': round(tgt_copy, 4),
        'distractor_copy_score': round(dis_copy, 4),
        'tgt_minus_dis_copy': round(tgt_copy - dis_copy, 4),
        'copy_score': round(copy_score, 4),
        'top_promoted': top_prom[:5],
        'top_suppressed': top_supp[:5],
    })

ov_df = pd.DataFrame(ov_rows)
print(ov_df[['label','target_copy_score','distractor_copy_score','tgt_minus_dis_copy','top_promoted']].to_string(index=False))

In [ ]:
# Bar chart: target vs distractor copy score per head, sorted by target−distractor
ov_sorted = ov_df.sort_values('tgt_minus_dis_copy', ascending=False)
labels_ov = ov_sorted['label'].values
tgt_cs    = ov_sorted['target_copy_score'].values
dis_cs    = ov_sorted['distractor_copy_score'].values

fig, ax = plt.subplots(figsize=(max(10, len(ov_sorted) * 0.7 + 2), 4))
x = np.arange(len(labels_ov))
ax.bar(x - 0.2, tgt_cs, 0.4, color='steelblue', label='Target pronoun copy score')
ax.bar(x + 0.2, dis_cs, 0.4, color='tomato',    label='Distractor pronoun copy score')
ax.set_xticks(x)
ax.set_xticklabels(labels_ov, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Copy score (OV logit when attending to name)')
ax.set_title('OV circuit: what each head writes when attending to gendered name\n(higher target score = Name-Mover)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_ov_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — Head Ablation Study

For each surviving head, zero its gate and re-run the test set.  
The accuracy **drop** tells us how load-bearing each head is.

This is different from the pruning objective: pruning removed structurally unimportant heads;
ablation here asks which of the *surviving* heads are most critical.

In [ ]:
# Baseline accuracy with the full circuit
baseline_res = run_evaluation(
    model_to_eval=circuit_model,
    model_name='Circuit-baseline',
    full_model_for_faithfulness=None,
    dataloader=test_dataloader,
    device=DEVICE,
    verbose=False,
    tokenizer=tokenizer,
)
baseline_acc = baseline_res['accuracy']
print(f'Circuit baseline accuracy: {baseline_acc:.4f}')

In [ ]:
# Ablate each surviving head and record accuracy
# WARNING: this runs one evaluation per head — can be slow for many heads
ablation_rows = []
for (l, h) in tqdm(surv_head_list, desc='Ablating heads'):
    acc = ablate_head_and_eval(circuit_model, l, h, test_dataloader, DEVICE)
    ablation_rows.append({
        'label': f'L{l}H{h}', 'layer': l, 'head': h,
        'acc_ablated': acc,
        'acc_drop': baseline_acc - acc,
    })
    print(f'  L{l}H{h}: {acc:.4f}  (drop {baseline_acc - acc:+.4f})')

abl_df = pd.DataFrame(ablation_rows).sort_values('acc_drop', ascending=False)
print('\nMost critical heads:')
print(abl_df.head(10).to_string(index=False))

In [ ]:
# Bar chart: accuracy drop per head
fig, ax = plt.subplots(figsize=(max(8, len(abl_df) * 0.6 + 2), 5))
colors_abl = ['tomato' if v > 0.01 else 'steelblue' for v in abl_df['acc_drop'].values]
ax.bar(range(len(abl_df)), abl_df['acc_drop'].values, color=colors_abl)
ax.set_xticks(range(len(abl_df)))
ax.set_xticklabels(abl_df['label'].values, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Head')
ax.set_ylabel('Accuracy drop')
ax.set_title(f'Head ablation — accuracy drop from circuit baseline ({baseline_acc:.3f})\n(red = critical head, blue = redundant)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Merge all head-level info into one summary table
if not role_df.empty and not abl_df.empty and not ov_df.empty:
    merged = role_df.merge(abl_df[['label','acc_ablated','acc_drop']], on='label', how='left')
    merged = merged.merge(ov_df[['label','target_copy_score','distractor_copy_score','tgt_minus_dis_copy']], on='label', how='left')
    merged = merged.sort_values('acc_drop', ascending=False)
    print('Head summary (sorted by ablation impact):')
    cols = ['label','role','name_attn','dla_diff','target_copy_score','tgt_minus_dis_copy','acc_drop']
    print(merged[cols].to_string(index=False))

---
## Section 7 — Logit Lens

Apply the final LayerNorm + unembedding to the residual stream at every layer.
This shows how the model's prediction *evolves* through the discovered circuit.

For GP: when does the residual stream first prefer `she` over `he` (or vice versa)?  
An early preference shift suggests the name-gender information enters the residual stream early.

In [ ]:
print('Computing logit lens ...')
N_LENS = min(32, N_ANALYSIS)
lens_batch    = {k: v[:N_LENS] for k, v in analysis_batch.items() if isinstance(v, torch.Tensor)}
lens_pred_pos = pred_pos[:N_LENS]

lens_result = compute_logit_lens(
    circuit_model,
    lens_batch['input_ids'],
    lens_batch['attention_mask'],
    lens_batch['corrupted_input_ids'],
    pred_positions=lens_pred_pos,
    device=DEVICE,
    topk=10,
)
print(f'Logit lens computed for {len(lens_result)} layer checkpoints.')

In [ ]:
# Capture target and distractor logit values at every layer (example 1)
target_id_eg     = analysis_batch['target_token'][1].item()
distractor_id_eg = analysis_batch['distractor_token'][1].item()
target_str       = tokenizer.decode([target_id_eg]).strip()
distractor_str   = tokenizer.decode([distractor_id_eg]).strip()

print(f'Target pronoun   : "{target_str}" (id={target_id_eg})')
print(f'Distractor pronoun: "{distractor_str}" (id={distractor_id_eg})')
print()

# Re-run with hooks to collect per-layer residual streams
_residuals_gp: dict = {}
_hooks = []
_embed_done = [False]

def _emb_hook(module, inp, out):
    if not _embed_done[0]:
        _residuals_gp[-1] = out.detach().cpu()
        _embed_done[0] = True
_hooks.append(circuit_model.model.embed_tokens.register_forward_hook(_emb_hook))

for _l, _layer in enumerate(circuit_model.model.layers):
    def _make_h(idx):
        def _h(module, inp, out):
            hs = out[0] if isinstance(out, tuple) else out
            _residuals_gp[idx] = hs.detach().cpu()
        return _h
    _hooks.append(_layer.register_forward_hook(_make_h(_l)))

circuit_model.eval()
with torch.no_grad():
    circuit_model(
        input_ids=lens_batch['input_ids'].to(DEVICE),
        attention_mask=lens_batch['attention_mask'].to(DEVICE),
        corrupted_input_ids=lens_batch['corrupted_input_ids'].to(DEVICE),
    )
for _h in _hooks:
    _h.remove()

# Decode logits for target and distractor at each layer
final_ln = circuit_model.model.norm
lm_head  = circuit_model.lm_head
B_lens   = lens_batch['input_ids'].shape[0]

_tgt_logits_per_layer = {}
_dis_logits_per_layer = {}

for l_idx in sorted(_residuals_gp.keys()):
    res = _residuals_gp[l_idx]  # (B, S, H)
    res_pred = torch.stack([res[b, lens_pred_pos[b]] for b in range(B_lens)]).to(DEVICE)
    with torch.no_grad():
        logits = lm_head(final_ln(res_pred)).float()  # (B, vocab)
    _tgt_logits_per_layer[l_idx] = logits[:, target_id_eg].cpu()
    _dis_logits_per_layer[l_idx] = logits[:, distractor_id_eg].cpu()

# Print table
print(f"{'Layer':<8} {'Tgt logit':>11}  {'Dis logit':>11}  {'Tgt − Dis':>11}")
print('-' * 48)
for l_idx in sorted(_tgt_logits_per_layer.keys()):
    tgt_l = _tgt_logits_per_layer[l_idx].mean().item()
    dis_l = _dis_logits_per_layer[l_idx].mean().item()
    label = 'emb' if l_idx == -1 else f'L{l_idx}'
    print(f'{label:<8} {tgt_l:>+11.3f}  {dis_l:>+11.3f}  {tgt_l - dis_l:>+11.3f}')

In [ ]:
# Plot target vs distractor logit across layers
layers_sorted = sorted(_tgt_logits_per_layer.keys())
layer_labels  = ['emb' if l == -1 else f'L{l}' for l in layers_sorted]
x3 = np.arange(len(layer_labels))

tgt_mean = np.array([_tgt_logits_per_layer[l].mean().item() for l in layers_sorted])
dis_mean = np.array([_dis_logits_per_layer[l].mean().item() for l in layers_sorted])
tgt_std  = np.array([_tgt_logits_per_layer[l].std().item()  for l in layers_sorted])
dis_std  = np.array([_dis_logits_per_layer[l].std().item()  for l in layers_sorted])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(x3, tgt_mean, 'o-', color='steelblue', linewidth=2, label=f'Target "{target_str}" logit')
ax.fill_between(x3, tgt_mean - tgt_std, tgt_mean + tgt_std, color='steelblue', alpha=0.15)
ax.plot(x3, dis_mean, 's--', color='tomato',    linewidth=2, label=f'Distractor "{distractor_str}" logit')
ax.fill_between(x3, dis_mean - dis_std, dis_mean + dis_std, color='tomato',    alpha=0.15)

ax.fill_between(x3, tgt_mean, dis_mean,
                where=(tgt_mean > dis_mean), color='steelblue', alpha=0.10, label='Target > Distractor')
ax.fill_between(x3, tgt_mean, dis_mean,
                where=(tgt_mean < dis_mean), color='tomato',    alpha=0.10, label='Distractor > Target')
ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax.set_xticks(x3)
ax.set_xticklabels(layer_labels, rotation=60, ha='right', fontsize=7)
ax.set_ylabel('Logit value (mean ± std)')
ax.set_title(f'Logit lens: target vs distractor pronoun through every layer\n'
             f'(n={B_lens}, shaded = dominant token region)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_logit_lens.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

Plots saved to `SAVE_DIR`:

| File | Content |
|------|---------|
| `posthoc_head_mask.png` | Surviving head heatmap |
| `posthoc_mlp_neurons.png` | MLP neuron survival per layer |
| `posthoc_attn_name.png` | Attention to name position per head |
| `posthoc_head_dla.png` | Per-head DLA |
| `posthoc_mlp_dla.png` | Per-layer MLP DLA |
| `posthoc_role_scatter.png` | Head role scatter |
| `posthoc_ov_scores.png` | OV copy scores |
| `posthoc_ablation.png` | Head ablation accuracy drop |
| `posthoc_logit_lens.png` | Logit lens |
| `posthoc_circuit_story.png` | End-to-end GP circuit story |

---
## Section 8 — Individual Neuron Analysis

What are the surviving neurons *actually doing*?  
We study three things:

| Analysis | What it reveals |
|----------|-----------------|
| **8.1 Vocab projection** | What each neuron *writes* to the residual stream — projected through W_U to vocabulary space |
| **8.2 Input sensitivity** | Which input tokens *activate* each neuron (gate path in SwiGLU) |
| **8.3 Activation on test data** | When does the neuron fire — correlation with target−distractor logit diff |

**Llama SwiGLU notation:**
$$\text{neuron}_i(x) = \sigma_{\text{silu}}(W_\text{gate}[i,:] \cdot x) \times W_\text{up}[i,:] \cdot x$$
$$\text{MLP output} = \sum_i W_\text{down}[:,i] \cdot \text{neuron}_i(x)$$

In [ ]:
import importlib
import posthoc_analysis.analysis_utils as _au
importlib.reload(_au)

from posthoc_analysis.analysis_utils import (
    get_mlp_neuron_vocab_projections,
    capture_mlp_neuron_activations,
    compute_neuron_io_correlation,
    capture_logit_diffs_batch,
)

# ---- Which layers to study ----
surv_neurons_all = get_surviving_mlp_neurons(circuit_model)
n_layers         = len(circuit_model.model.layers)

mlp_ranked = sorted(
    [(l, len(surv_neurons_all[l][0])) for l in range(n_layers) if len(surv_neurons_all[l][0]) > 0],
    key=lambda x: -x[1]
)
NEURON_LAYERS = [l for l, _ in mlp_ranked[:3]] if mlp_ranked else [0]
TOP_N    = 32
MAX_NEURONS = 20

# Filter to high-activating subset
high_act_neurons = {}   # {layer: sorted list of neuron indices}

for LAYER in NEURON_LAYERS:
    all_surv = surv_neurons_all[LAYER][0].tolist()
    acts = capture_mlp_neuron_activations(
        circuit_model,
        analysis_batch['input_ids'],
        analysis_batch['attention_mask'],
        analysis_batch['corrupted_input_ids'],
        layer=LAYER,
        pred_positions=pred_pos,
        device=DEVICE,
    )  # (B, intermediate_size)

    mean_act = acts[:, all_surv].abs().mean(0)    # (n_surv,)
    top_local  = mean_act.topk(min(TOP_N, len(all_surv))).indices.tolist()
    top_global = [all_surv[i] for i in top_local]
    high_act_neurons[LAYER] = top_global

    print(f"L{LAYER}: {len(all_surv)} surviving → keeping top-{len(top_global)} by mean |activation|")

### 8.1 Vocab projection — what does each neuron write?

For each surviving neuron $i$, we compute $W_U \cdot W_\text{down}[:,i]$ and report the top
promoted / suppressed tokens.  A neuron that promotes `she`/`her` or `he`/`his` is
likely part of the gender-pronoun pathway.

In [ ]:
vocab_proj_all = {}   # {layer: {neuron_idx: dict}}

for LAYER in NEURON_LAYERS:
    neuron_subset = high_act_neurons[LAYER]
    print(f"\nLayer {LAYER}: vocab projections for {len(neuron_subset)} high-activating neurons ...")
    proj = get_mlp_neuron_vocab_projections(
        circuit_model, LAYER, neuron_subset, tokenizer, top_k=10
    )
    vocab_proj_all[LAYER] = proj

    proj_sorted = sorted(proj.items(), key=lambda kv: -kv[1]["write_norm"])

    print(f"\n{'Neuron':>8}  {'Write‖':>8}  {'Tgt wl':>8}  {'Dis wl':>8}  {'Tgt−Dis':>7}  {'Top-3 promoted':28}  Top-3 suppressed")
    print("-" * 110)
    for idx, info in proj_sorted[:20]:
        prom = " | ".join(f'"{t}"' for t in info["top_promoted"][:3])
        supp = " | ".join(f'"{t}"' for t in info["top_suppressed"][:3])
        tgt_wl = torch.stack([
            info["write_logits"][target_ids[b].item()]
            for b in range(len(target_ids))
        ]).float().mean().item()
        dis_wl = torch.stack([
            info["write_logits"][distractor_ids[b].item()]
            for b in range(len(distractor_ids))
        ]).float().mean().item()
        print(f"  N{idx:>5d}  {info['write_norm']:>8.3f}  {tgt_wl:>+8.3f}  {dis_wl:>+8.3f}  {tgt_wl-dis_wl:>+7.3f}  {prom:28s}  {supp}")

In [ ]:
# Heatmap: target vs distractor write-logit for top neurons in each layer
for LAYER in NEURON_LAYERS:
    proj = vocab_proj_all[LAYER]
    proj_sorted  = sorted(proj.items(), key=lambda kv: -kv[1]["write_norm"])
    plot_neurons = [idx for idx, _ in proj_sorted[:MAX_NEURONS]]

    tgt_scores_n = []
    dis_scores_n = []

    for idx in plot_neurons:
        wl = proj[idx]["write_logits"]
        tgt_w = torch.stack([wl[target_ids[b].item()]     for b in range(len(target_ids))]).mean().item()
        dis_w = torch.stack([wl[distractor_ids[b].item()] for b in range(len(distractor_ids))]).mean().item()
        tgt_scores_n.append(tgt_w)
        dis_scores_n.append(dis_w)

    diff_n = np.array(tgt_scores_n) - np.array(dis_scores_n)
    order  = np.argsort(-diff_n)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(plot_neurons) * 0.18 + 1)))

    y = np.arange(len(plot_neurons))
    ax = axes[0]
    ax.barh(y,       np.array(tgt_scores_n)[order], 0.4, label='Target write-logit',     color='steelblue')
    ax.barh(y + 0.4, np.array(dis_scores_n)[order], 0.4, label='Distractor write-logit', color='tomato')
    ax.set_yticks(y + 0.2)
    ax.set_yticklabels([f'N{plot_neurons[i]}' for i in order], fontsize=7)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Write-logit value')
    ax.set_title(f'L{LAYER} MLP neurons: target vs distractor write-logit')
    ax.legend(fontsize=8)

    ax2 = axes[1]
    colors_n = ['steelblue' if v >= 0 else 'tomato' for v in diff_n[order]]
    ax2.barh(y, diff_n[order], color=colors_n)
    ax2.set_yticks(y)
    ax2.set_yticklabels([f'N{plot_neurons[i]}' for i in order], fontsize=7)
    ax2.axvline(0, color='black', linewidth=0.8)
    ax2.set_xlabel('Target − Distractor write-logit')
    ax2.set_title(f'L{LAYER} MLP neurons: task-relevant write direction')

    plt.suptitle(f'Layer {LAYER} — MLP neuron vocab projections (top {len(plot_neurons)} by write-norm)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'posthoc_mlp_neuron_vocab_L{LAYER}.png'), dpi=150)
    plt.show()

### 8.2 Input sensitivity — what activates each neuron?

`gate_proj[i, :] @ W_E[t, :]` tells you how much token $t$ drives neuron $i$'s gate pathway.

Neurons with high sensitivity to gender-indicative names (e.g. `Diamond`, `Tyler`) are
*gender detectors*; those sensitive to verbs/prepositions are syntactic neurons.

In [ ]:
for LAYER in NEURON_LAYERS:
    proj = vocab_proj_all[LAYER]
    proj_sorted = sorted(proj.items(), key=lambda kv: -kv[1]["write_norm"])
    top20 = proj_sorted[:20]

    mlp_g = circuit_model.model.layers[LAYER].mlp.original_mlp
    W_g   = mlp_g.gate_proj.weight.detach().float().cpu()   # (intermediate, hidden)
    W_E_g = circuit_model.model.embed_tokens.weight.detach().float().cpu()  # (vocab, hidden)

    print(f"\n=== Layer {LAYER} — Input sensitivity for target and distractor pronouns ===\n")
    print(f"{'Neuron':>8}  {'Tgt gate':>9}  {'Dis gate':>9}  {'Tgt−Dis gate':>12}  {'Tgt wl':>8}  {'Dis wl':>8}  Top-4 gate activating")
    print("-" * 110)
    for idx, info in top20:
        tgt_gate = torch.stack([
            W_g[idx] @ W_E_g[target_ids[b].item()]
            for b in range(len(target_ids))
        ]).mean().item()
        dis_gate = torch.stack([
            W_g[idx] @ W_E_g[distractor_ids[b].item()]
            for b in range(len(distractor_ids))
        ]).mean().item()
        tgt_wl = torch.stack([
            info["write_logits"][target_ids[b].item()]
            for b in range(len(target_ids))
        ]).float().mean().item()
        dis_wl = torch.stack([
            info["write_logits"][distractor_ids[b].item()]
            for b in range(len(distractor_ids))
        ]).float().mean().item()
        gate_str = " | ".join(f'"{t}"' for t in info.get("top_gate_activating", [])[:4])
        print(f"  N{idx:>5d}  {tgt_gate:>+9.3f}  {dis_gate:>+9.3f}  {tgt_gate-dis_gate:>+12.3f}  {tgt_wl:>+8.3f}  {dis_wl:>+8.3f}  {gate_str}")

### 8.3 Activation on test data — when do neurons fire?

We correlate each surviving neuron's activation at the prediction position with the
target−distractor logit difference.

- **High positive correlation** → neuron fires more when model confidently predicts the correct pronoun
- **High negative correlation** → neuron fires more on wrong predictions (possibly confounding)

In [ ]:
# Collect target−distractor logit diffs over the full test set
print('Collecting target−distractor logit diffs over test set ...')
logit_diffs_all, correct_all = capture_logit_diffs_batch(circuit_model, test_dataloader, DEVICE)
print(f'  Mean logit diff: {logit_diffs_all.mean():.3f}  |  Accuracy: {correct_all.float().mean():.3f}')

In [ ]:
for LAYER in NEURON_LAYERS:
    neuron_subset = high_act_neurons[LAYER]

    acts = capture_mlp_neuron_activations(
        circuit_model,
        analysis_batch['input_ids'],
        analysis_batch['attention_mask'],
        analysis_batch['corrupted_input_ids'],
        layer=LAYER,
        pred_positions=pred_pos,
        device=DEVICE,
    )  # (B, intermediate_size)
    acts_sub = acts[:, neuron_subset]   # (B, TOP_N)

    # Target−distractor logit diff for this batch
    with torch.no_grad():
        out = circuit_model(
            input_ids=analysis_batch['input_ids'].to(DEVICE),
            attention_mask=analysis_batch['attention_mask'].to(DEVICE),
            corrupted_input_ids=analysis_batch['corrupted_input_ids'].to(DEVICE),
        )
    logits = out.logits if hasattr(out, 'logits') else out[0]
    B_nb = logits.shape[0]
    ld_batch = torch.stack([
        logits[b, pred_pos[b], target_ids[b]] - logits[b, pred_pos[b], distractor_ids[b]]
        for b in range(B_nb)
    ]).cpu()

    corr     = compute_neuron_io_correlation(acts_sub, ld_batch).float()  # (TOP_N,)
    mean_act = acts_sub.abs().mean(0).float()                              # (TOP_N,)
    surv_arr = np.array(neuron_subset)

    top_pos = corr.topk(min(10, len(corr))).indices.tolist()
    top_neg = corr.topk(min(10, len(corr)), largest=False).indices.tolist()

    print(f"\n=== Layer {LAYER} — top task-correlated neurons ===")
    for si in top_pos:
        info = vocab_proj_all[LAYER].get(surv_arr[si], {})
        print(f"  N{surv_arr[si]:5d}  corr={corr[si]:+.3f}  act={mean_act[si]:.3f}  writes: {info.get('top_promoted',[])[:3]}")

    print(f"\n=== Layer {LAYER} — top anti-correlated neurons ===")
    for si in top_neg:
        info = vocab_proj_all[LAYER].get(surv_arr[si], {})
        print(f"  N{surv_arr[si]:5d}  corr={corr[si]:+.3f}  act={mean_act[si]:.3f}  writes: {info.get('top_promoted',[])[:3]}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(corr.numpy(), bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_xlabel('Pearson r  (neuron activation vs target−distractor logit diff)')
    axes[0].set_ylabel('# neurons')
    axes[0].set_title(f'L{LAYER} neuron–task correlation')

    sc = axes[1].scatter(mean_act.numpy(), corr.numpy(),
                         c=corr.numpy(), cmap='RdBu', s=50,
                         vmin=-corr.abs().max().item(), vmax=corr.abs().max().item())
    for si in top_pos + top_neg:
        axes[1].annotate(f'N{surv_arr[si]}', (mean_act[si].item(), corr[si].item()), fontsize=7)
    axes[1].axhline(0, color='black', lw=0.8)
    plt.colorbar(sc, ax=axes[1], label='correlation')
    axes[1].set_xlabel('Mean |activation|')
    axes[1].set_ylabel('Correlation with target−distractor logit diff')
    axes[1].set_title(f'L{LAYER} activation strength vs task relevance')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'posthoc_neuron_activation_L{LAYER}.png'), dpi=150)
    plt.show()

---
## Section 9 — End-to-End GP Circuit Story

This section synthesises every analysis above into a single narrative:  
**How does the pruned circuit map a gendered name to the correct gender pronoun?**

For GP (unlike IOI's 3-stage circuit), the expected mechanism is simpler:
1. **Stage 1 — Name detection (early layers):** Some heads/neurons detect the gendered name
   and build a gender-signal in the residual stream.
2. **Stage 2 — Pronoun output (late layers):** Name-Mover heads and MLP neurons read the
   gender signal and write the correct pronoun logit at the prediction position.

In [ ]:
# ── Section 9 · End-to-End GP Circuit Story ───────────────────────────────
# Requires: role_df, head_dla_diff, mlp_dla_diff, _tgt_logits_per_layer,
#           _dis_logits_per_layer, surv_head_list  (all computed above)

ROLE_COLOR = {
    'Name-Mover':        '#2166ac',
    'Name-Suppressor':   '#4393c3',
    'Pronoun-Promoter':  '#2ca02c',
    'Pronoun-Suppressor':'#d6604d',
    'Other':             '#b0b0b0',
}

n_layers_story = len(circuit_model.model.layers)

import random
random.seed(42)

fig = plt.figure(figsize=(22, 14))
gs  = fig.add_gridspec(3, 1, hspace=0.45, height_ratios=[2.5, 1.5, 1.5])
ax1 = fig.add_subplot(gs[0])   # head role timeline
ax2 = fig.add_subplot(gs[1])   # cumulative DLA
ax3 = fig.add_subplot(gs[2])   # logit lens

# ── PANEL 1 — Head role timeline ──────────────────────────────────────────
_y_offset_counter = {}

for _, row in role_df.iterrows():
    l    = row['layer']
    role = row['role']
    dla  = abs(row['dla_diff'])
    color = ROLE_COLOR.get(role, '#b0b0b0')

    _y_offset_counter[l] = _y_offset_counter.get(l, 0)
    y_pos = _y_offset_counter[l]
    _y_offset_counter[l] += 1

    size = max(80, min(800, dla * 1200 + 80))
    ax1.scatter(l, y_pos, s=size, color=color, alpha=0.85, zorder=3,
                edgecolors='white', linewidths=0.8)
    ax1.annotate(row['label'], (l, y_pos), ha='center', va='center',
                 fontsize=6, color='white', fontweight='bold', zorder=4)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_patches = [Patch(color=c, label=r) for r, c in ROLE_COLOR.items()]
size_legend    = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='grey',
           markersize=ms, label=lbl)
    for ms, lbl in [(6, 'small |DLA|'), (12, 'large |DLA|')]
]
ax1.legend(handles=legend_patches + size_legend, loc='upper left', fontsize=8, ncol=3)

# Stage annotation bands (simplified: early = name detection, late = pronoun output)
stage_alpha = 0.07
mid = n_layers_story // 2
ax1.axvspan(-0.5, mid - 0.5,                 color='orange',    alpha=stage_alpha)
ax1.axvspan(mid - 0.5, n_layers_story - 0.5, color='steelblue', alpha=stage_alpha)
ax1.text(mid * 0.5,        -0.7, 'Stage 1\nName Detection',   color='#b05000', fontsize=8, ha='center', style='italic')
ax1.text(mid + mid * 0.5, -0.7, 'Stage 2\nPronoun Output',   color='#1a3c6e', fontsize=8, ha='center', style='italic')

ax1.set_xlim(-0.5, n_layers_story - 0.5)
ax1.set_xlabel('Layer', fontsize=10)
ax1.set_ylabel('Head (stacked per layer)', fontsize=9)
ax1.set_title('Panel 1 — GP Circuit Head Role Timeline\n'
              'Bubble = surviving head | Size = |DLA(target−distractor)| | Color = role', fontsize=11)
ax1.set_xticks(range(n_layers_story))
ax1.set_xticklabels([f'L{l}' for l in range(n_layers_story)], fontsize=7)
ax1.grid(axis='x', alpha=0.3)

# ── PANEL 2 — Cumulative DLA per layer ────────────────────────────────────
layer_attn_dla = {}
for (l, h), val in head_dla_diff.items():
    layer_attn_dla[l] = layer_attn_dla.get(l, 0.0) + val

layers_all = list(range(n_layers_story))
attn_vals  = [layer_attn_dla.get(l, 0.0) for l in layers_all]
mlp_vals   = [mlp_dla_diff.get(l, 0.0)   for l in layers_all]

attn_pos = [max(0, v) for v in attn_vals]
attn_neg = [min(0, v) for v in attn_vals]
mlp_pos  = [max(0, v) for v in mlp_vals]
mlp_neg  = [min(0, v) for v in mlp_vals]

x2 = np.arange(n_layers_story)
ax2.bar(x2, attn_pos, color='steelblue', alpha=0.85, label='Attn heads (target-promoting)')
ax2.bar(x2, attn_neg, color='tomato',    alpha=0.85, label='Attn heads (distractor-promoting)')
ax2.bar(x2, mlp_pos,  color='#2ca02c',  alpha=0.6,  label='MLP (target-promoting)',     bottom=attn_pos)
ax2.bar(x2, mlp_neg,  color='#d62728',  alpha=0.6,  label='MLP (distractor-promoting)', bottom=attn_neg)
ax2.axhline(0, color='black', linewidth=0.8)

surv_mlp_blks = get_surviving_mlp_blocks(circuit_model)
for l in layers_all:
    if not surv_mlp_blks.get(l, True):
        ax2.axvspan(l - 0.4, l + 0.4, color='grey', alpha=0.15, hatch='//')

ax2.set_xticks(x2)
ax2.set_xticklabels([f'L{l}' for l in layers_all], fontsize=7)
ax2.set_ylabel('DLA (target − distractor logit contribution)')
ax2.set_title('Panel 2 — DLA per Layer: How Each Layer Shifts the Target vs Distractor Decision\n'
              '(blue/green = promotes target pronoun | red = promotes distractor | hatched = pruned MLP)', fontsize=10)
ax2.legend(fontsize=8, ncol=2)
ax2.grid(axis='y', alpha=0.3)

# ── PANEL 3 — Logit lens ──────────────────────────────────────────────────
if '_tgt_logits_per_layer' in dir() or '_tgt_logits_per_layer' in globals():
    layers_sorted_s = sorted(_tgt_logits_per_layer.keys())
    layer_labels_s  = ['emb' if l == -1 else f'L{l}' for l in layers_sorted_s]
    x3 = np.arange(len(layer_labels_s))

    tgt_m = np.array([_tgt_logits_per_layer[l].mean().item() for l in layers_sorted_s])
    dis_m = np.array([_dis_logits_per_layer[l].mean().item() for l in layers_sorted_s])
    tgt_s = np.array([_tgt_logits_per_layer[l].std().item()  for l in layers_sorted_s])
    dis_s = np.array([_dis_logits_per_layer[l].std().item()  for l in layers_sorted_s])

    ax3.plot(x3, tgt_m, 'o-', color='steelblue', linewidth=2, label=f'Target "{target_str}" logit')
    ax3.fill_between(x3, tgt_m - tgt_s, tgt_m + tgt_s, color='steelblue', alpha=0.15)
    ax3.plot(x3, dis_m, 's--', color='tomato',    linewidth=2, label=f'Distractor "{distractor_str}" logit')
    ax3.fill_between(x3, dis_m - dis_s, dis_m + dis_s, color='tomato',    alpha=0.15)
    ax3.fill_between(x3, tgt_m, dis_m, where=(tgt_m > dis_m), color='steelblue', alpha=0.10, label='Target > Distractor')
    ax3.fill_between(x3, tgt_m, dis_m, where=(tgt_m < dis_m), color='tomato',    alpha=0.10, label='Distractor > Target')
    ax3.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax3.set_xticks(x3)
    ax3.set_xticklabels(layer_labels_s, rotation=60, ha='right', fontsize=7)
    ax3.set_ylabel('Logit value (mean ± std)')
    ax3.set_title(f'Panel 3 — Logit Lens: Target vs Distractor Pronoun Through Every Layer', fontsize=10)
    ax3.legend(fontsize=8, ncol=2)
    ax3.grid(True, alpha=0.3)

plt.suptitle(
    'End-to-End GP Circuit Story: How the Pruned Circuit Maps Gendered Name → Correct Pronoun\n'
    f'Circuit: {run_config["model"]}  |  KL budget: {run_config["kl_budget"]}  |  '
    f'{len(surv_head_list)} surviving heads',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig(os.path.join(SAVE_DIR, 'posthoc_circuit_story.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved -> {SAVE_DIR}/posthoc_circuit_story.png')

In [ ]:
# ── Section 9 · Printed narrative summary ─────────────────────────────────
print("=" * 72)
print("  END-TO-END GP CIRCUIT NARRATIVE")
print("=" * 72)

eg_sent     = tokenizer.decode(analysis_batch['input_ids'][0].tolist(), skip_special_tokens=True)
tgt_name    = target_str
dis_name    = distractor_str
eg_name_raw = test_dataset.processed_data[0].get('name', '<unknown>')

print(f"\nExample sentence (example 0):")
print(f"  {eg_sent[:120]}...")
print(f"\n  Gendered name    : '{eg_name_raw}'")
print(f"  Target pronoun   : '{tgt_name}'")
print(f"  Distractor pronoun: '{dis_name}'")

if not role_df.empty:
    mid_cutoff = n_layers_story // 2

    early_heads  = role_df[role_df['layer'] < mid_cutoff].sort_values('name_attn', ascending=False)
    late_heads   = role_df[(role_df['layer'] >= mid_cutoff) & (role_df['role'] == 'Name-Mover')]
    promoters    = role_df[role_df['role'] == 'Pronoun-Promoter'].sort_values('dla_diff', ascending=False)

    print(f"\n{'─'*60}")
    print(f"STAGE 1 · Early layers (L0–L{mid_cutoff-1}): Name Detection")
    print(f"{'─'*60}")
    if not early_heads.empty:
        for _, r in early_heads.head(5).iterrows():
            print(f"  {r['label']:8s}  name_attn={r['name_attn']:.3f}  DLA(tgt−dis)={r['dla_diff']:+.3f}  role={r['role']}")
        print(f"  → Early heads with high name attention may build a gender signal")
        print(f"    in the residual stream that later layers use to predict the pronoun.")
    else:
        print("  (no surviving heads in early layers)")

    print(f"\n{'─'*60}")
    print(f"STAGE 2 · Late layers (L{mid_cutoff}–L{n_layers_story-1}): Pronoun Output")
    print(f"{'─'*60}")
    if not late_heads.empty:
        for _, r in late_heads.iterrows():
            ov_row = ov_df[ov_df['label'] == r['label']]
            tgt_copy = ov_row['target_copy_score'].values[0] if not ov_row.empty else float('nan')
            print(f"  {r['label']:8s}  name_attn={r['name_attn']:.3f}  DLA={r['dla_diff']:+.3f}  OV tgt_copy={tgt_copy:+.3f}")
        print(f"  → Name-Mover heads in late layers attend to '{eg_name_raw}' and write")
        print(f"    a direction that promotes the correct pronoun '{tgt_name}'.")
    elif not promoters.empty:
        for _, r in promoters.head(5).iterrows():
            print(f"  {r['label']:8s}  DLA(tgt−dis)={r['dla_diff']:+.3f}  role={r['role']}")
        print(f"  → Pronoun-Promoter heads push towards the target pronoun globally.")
    else:
        print("  (no Name-Mover or Pronoun-Promoter heads in late layers — check thresholds)")

    # Final tally
    n_name_movers   = (role_df['role'] == 'Name-Mover').sum()
    n_pron_promoters = (role_df['role'] == 'Pronoun-Promoter').sum()
    n_other         = (role_df['role'] == 'Other').sum()
    final_tgt_logit = _tgt_logits_per_layer[max(_tgt_logits_per_layer.keys())].mean().item()
    final_dis_logit = _dis_logits_per_layer[max(_dis_logits_per_layer.keys())].mean().item()

    print(f"\n{'='*60}")
    print(f"  GP CIRCUIT SUMMARY")
    print(f"{'='*60}")
    print(f"  Name-Mover heads       : {n_name_movers}")
    print(f"  Pronoun-Promoter heads : {n_pron_promoters}")
    print(f"  Other heads            : {n_other}")
    print(f"  Circuit accuracy       : {baseline_acc:.4f}")
    print(f"  Final target logit     : {final_tgt_logit:+.3f}")
    print(f"  Final distractor logit : {final_dis_logit:+.3f}")
    print(f"  Final tgt − dis        : {final_tgt_logit - final_dis_logit:+.3f}  "
          f"({'CORRECT' if final_tgt_logit > final_dis_logit else 'INCORRECT'})")